# PathSnap First Review — LightGBM Candidate Classifier
**Author**: PES2UG24CS064 (Aniket S Patil)  
**Algorithm Family**: Boosting (Gradient Boosted Decision Trees)

In [ ]:
from pathlib import Path
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, log_loss, confusion_matrix
import matplotlib.pyplot as plt

# Locate shared dataset
ROOT = Path.cwd().parent if Path.cwd().name == 'PES2UG24CS064_LightGBM' else Path.cwd()
DATA_DIR = ROOT / 'PES2UG24CS102_RandomForest' / 'shared_data'
if not DATA_DIR.exists():
    DATA_DIR = ROOT / 'data'
print(f'Data directory: {DATA_DIR}')

In [ ]:
# Load train and student-test datasets
train = np.load(DATA_DIR / 'train_candidates.npz')
test = np.load(DATA_DIR / 'student_test_candidates.npz')

X_tr, y_tr = train['X'], train['y']
trip_tr, obs_tr = train['trip_id'], train['gps_observation_id']
features = [str(f) for f in train['feature_names']]

X_te, y_te = test['X'], test['y']
obs_te = test['gps_observation_id']

print(f'Features ({len(features)}): {features}')
print(f'Train candidate rows: {len(X_tr):,} | Positive class: {np.mean(y_tr)*100:.2f}%')
print(f'Student-test rows:    {len(X_te):,} | Positive class: {np.mean(y_te)*100:.2f}%')

In [ ]:
# Helper for observation-level Top-1 Accuracy
def calc_top1(obs_ids, y_true, probs):
    best = {}
    for i, oid in enumerate(obs_ids):
        p = probs[i]
        if oid not in best or p > best[oid][0]:
            best[oid] = (p, y_true[i])
    return float(np.mean([label == 1 for p, label in best.values()]) * 100.0)

In [ ]:
# 5-Fold Group-Aware Cross-Validation (by trip_id)
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 150,
    'learning_rate': 0.08,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

gkf = GroupKFold(n_splits=5)
cv_f1s, cv_top1s, cv_losses = [], [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_tr, y_tr, groups=trip_tr), 1):
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    probs = m.predict_proba(X_tr[va_idx])[:, 1]
    preds = (probs >= 0.5).astype(int)
    
    f1 = f1_score(y_tr[va_idx], preds, average='macro')
    ll = log_loss(y_tr[va_idx], probs)
    top1 = calc_top1(obs_tr[va_idx], y_tr[va_idx], probs)
    cv_f1s.append(f1); cv_top1s.append(top1); cv_losses.append(ll)
    print(f'Fold {fold} | Macro F1: {f1:.4f} | Top-1: {top1:.2f}% | Log Loss: {ll:.4f}')

print(f'\nCV Macro F1:       {np.mean(cv_f1s):.4f} (+/- {np.std(cv_f1s):.4f})')
print(f'CV Top-1 Accuracy: {np.mean(cv_top1s):.2f}% (+/- {np.std(cv_top1s):.2f}%)')
print(f'CV Log Loss:       {np.mean(cv_losses):.4f}')

In [ ]:
# Fit Final Model on 100% Training Data & Evaluate on Student-Testing Partition
model = lgb.LGBMClassifier(**lgb_params)
model.fit(X_tr, y_tr)

test_probs = model.predict_proba(X_te)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

top1 = calc_top1(obs_te, y_te, test_probs)
macro_f1 = f1_score(y_te, test_preds, average='macro')
prec = precision_score(y_te, test_preds)
rec = recall_score(y_te, test_preds)
ll = log_loss(y_te, test_probs)

print('=== STUDENT-TEST EVALUATION RESULTS ===')
print(f'Top-1 Observation Accuracy: {top1:.2f}%')
print(f'Macro F1-Score:             {macro_f1:.4f}')
print(f'Precision:                  {prec:.4f}')
print(f'Recall:                     {rec:.4f}')
print(f'Log Loss:                   {ll:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_te, test_preds))

In [ ]:
# Feature Importance Plot
importances = model.feature_importances_
order = np.argsort(importances)

plt.figure(figsize=(8, 4))
plt.barh(np.array(features)[order], importances[order], color='#1f77b4')
plt.title('LightGBM Feature Importances (Split Frequency)')
plt.xlabel('Number of Splits')
plt.tight_layout()
plt.show()